# Three-Country Phase-Diagram Placement Merge (UK + AU + DE) -- Full Draft for the Paper's Flagship Phase Figure

**Theory line**: multiplicative correction helps iff rho > sigma_c / (2*sigma_r) (derived
in log space in the UK's equivalent analysis). Coordinates = station-level
**(sigma_ratio, rho)**, where rho = corr(log F, log rho_resid), sigma_ratio =
std(log F) / std(log rho_resid) (ddof=0; the same convention across the UK, AU, and DE
scripts).

**Data sources** (all read-only):
- `results/three_country_phase_points.csv` -- UK 240 + AU 180 + DE 15 = 435 placements
  (UK/AU are transcribed from `uk_au_phase_points_merged.csv`; DE is computed by
  `021_de_phase_points.py`);
- `results/de_phase_summary.json` -- confusion matrix, placement verification, and
  small-sample registration.

**German-specific caveat**: single region (Börde), 13 stations -- each DE placement's rho
is estimated from a sample of only 13 stations (Fisher z SE ~= 0.316), so sampling error
is large; this is registered as-is rather than used for statistical inference (per the
paper's Germany-section convention); the GNN base is the training region's own prediction
(no held-out test fold), differing from the UK/AU test-fold convention (see deviation
record item 1 in the DE script).

In [ ]:
# Environment setup: cap CPU threads at 4 (GPU is running a tau sweep elsewhere); font config; data loading
import os
for _v in ("OMP_NUM_THREADS", "MKL_NUM_THREADS",
           "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ.setdefault(_v, "4")

import json
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Font config (minus-sign fix)
mpl.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
mpl.rcParams["axes.unicode_minus"] = False

DE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RESULTS_DIR = DE_DIR / "results"

three = pd.read_csv(RESULTS_DIR / "three_country_phase_points.csv",
                    encoding="utf-8-sig", dtype={"seed": str, "fold": str})
summary = json.loads((RESULTS_DIR / "de_phase_summary.json")
                     .read_text(encoding="utf-8"))

assert len(three) == 435, f"row count {len(three)} != 435 (UK 240 + AU 180 + DE 15)"
print("Placement count by country x base type:")
print(three.groupby(["country", "base_type"]).size().unstack(fill_value=0))

## Combined Three-Country Phase Diagram (Flagship Figure Draft)

- **Country color coding** (Okabe-Ito colorblind-safe palette, fixed order): UK blue
  `#0072B2` / AU teal `#009E73` / DE vermillion `#D55E00`;
- **Secondary encoding**: base type is encoded by marker shape (static = circle,
  GNN = triangle) -- identity does not rely on color alone;
- DE points are drawn larger with a black outline to stand out (the 15 points added in
  this step);
- Above the theory line rho = sigma_ratio/2 is the help region (correction predicted to
  improve accuracy), below it is the hurt region (predicted to worsen it);
- **Clamping of near-zero sigma_ratio points**: some of AU's x N combinations fall in
  NTL-saturated urban areas (sigma_c ~= 0, sigma_ratio as low as 1e-16); to preserve
  overall readability, points with sigma_ratio < 0.008 are clamped and drawn at the left
  edge dashed line (the count is tallied by rule and annotated on the figure; clamping
  does not change any point's visual help/hurt classification -- the |rho| of every
  clamped point is far from the local threshold of 0.004).

In [ ]:
# Country color coding (Okabe-Ito colorblind-safe; passes all six validate_palette checks) -- fixed order, never reorder by filtering
COUNTRY_COLOR = {"UK": "#0072B2", "AU": "#009E73", "DE": "#D55E00"}
COUNTRY_ORDER = ["UK", "AU", "DE"]
MARKER_OF = {"static": ("o", "Static base"), "gnn": ("^", "GNN base")}

# Clamping of near-zero sigma_ratio points (NTL-saturated urban areas, sigma_c ~= 0; count tallied by rule)
X_CLIP = 8e-3
clipped = three[three["sigma_ratio"] < X_CLIP]
n_clip = int(len(clipped))
# Premise for not changing the visual classification: every clamped point's |rho| is far from the local threshold X_CLIP/2
assert (clipped["pearson_log"].abs() > X_CLIP / 2).all() or n_clip == 0
clip_countries = ", ".join(sorted(clipped["country"].unique()))
clip_signals = ", ".join(sorted(clipped["signal"].unique()))

x_plot = np.clip(three["sigma_ratio"].values, X_CLIP, None)
x_hi = float(three["sigma_ratio"].max()) * 1.2
x_lo = X_CLIP * 0.82

fig, ax = plt.subplots(figsize=(9.0, 6.4), dpi=150)

# Theory line and help/hurt region shading (kept light so it doesn't compete with the data for attention)
xs = np.geomspace(x_lo, x_hi, 400)
ax.plot(xs, xs / 2, color="#444444", lw=1.6, zorder=3,
        label="Theory line ρ = σ_ratio / 2")
ax.fill_between(xs, xs / 2, 1.08, color="#009E73", alpha=0.055, zorder=0)
ax.fill_between(xs, -1.08, xs / 2, color="#D55E00", alpha=0.055, zorder=0)

plot_df = three.assign(_x=x_plot)
for country in COUNTRY_ORDER:
    sub = plot_df[plot_df["country"] == country]
    for kind, (marker, kind_label) in MARKER_OF.items():
        mask = (sub["base_type"] == "gnn") if kind == "gnn" \
            else sub["base_type"].isin(["uniform", "gpm"])
        blk = sub[mask]
        is_de = country == "DE"
        ax.scatter(blk["_x"], blk["pearson_log"], marker=marker,
                   s=80 if is_de else 26,
                   facecolor=COUNTRY_COLOR[country],
                   edgecolor="black" if is_de else "white",
                   linewidths=0.9 if is_de else 0.4,
                   alpha=0.95 if is_de else 0.72,
                   zorder=5 if is_de else 2,
                   label=f"{country} · {kind_label}")

# Clamp-boundary dashed line + rule-generated annotation (placed in the blank band above the legend, white background for readability)
_NOTE_BOX = dict(boxstyle="round,pad=0.3", fc="white", ec="none", alpha=0.78)
if n_clip > 0:
    ax.axvline(X_CLIP, color="#999999", lw=0.9, ls="--", zorder=1)
    ax.text(X_CLIP * 1.15, -0.30,
            f"{n_clip} points with σ_ratio < {X_CLIP:g} are clamped here\n"
            f"({clip_countries} x {clip_signals}: NTL-saturated urban areas, σ_c ≈ 0)",
            fontsize=8, color="#555555", va="top", bbox=_NOTE_BOX, zorder=6)

# Region labels (text uses a neutral text color, not a series color)
ax.text(x_lo * 1.25, 0.97, "help region (predicted improvement)", fontsize=10,
        color="#33413c", va="top")
ax.text(x_hi * 0.92, -0.97, "hurt region (predicted worsening)", fontsize=10,
        color="#4a3a31", va="bottom", ha="right")

# DE key-placement annotations: GNN x NP (the +193% worsening arm) and static x NP (the synergistic arm)
de = plot_df[plot_df["country"] == "DE"]
de_gnn_np = de[(de["base_type"] == "gnn") & (de["signal"] == "NP")]
de_static_np = de[(de["base_type"].isin(["uniform", "gpm"]))
                  & (de["signal"] == "NP")]
ax.annotate("DE GNN x NP (3 seeds, +193% worse)",
            xy=(float(de_gnn_np["_x"].median()),
                float(de_gnn_np["pearson_log"].median())),
            xytext=(0.42, -0.42), fontsize=9, color="#222222",
            bbox=_NOTE_BOX, zorder=6,
            arrowprops=dict(arrowstyle="->", color="#666666", lw=0.9))
ax.annotate("DE static x NP (synergistic)",
            xy=(float(de_static_np["_x"].median()),
                float(de_static_np["pearson_log"].median())),
            xytext=(0.055, 0.44), fontsize=9, color="#222222",
            bbox=_NOTE_BOX, zorder=6,
            arrowprops=dict(arrowstyle="->", color="#666666", lw=0.9))

ax.set_xscale("log")
ax.set_xlim(x_lo, x_hi)
ax.set_ylim(-1.08, 1.08)
ax.set_xlabel("σ_ratio = σ_c / σ_r (log scale)")
ax.set_ylabel("ρ = corr(log F, log ρ_resid)")
ax.set_title("Three-country phase-diagram placements (help criterion: ρ > σ_ratio / 2)\n"
             "UK 240 + AU 180 + DE 15 = 435 (base, region, signal) combinations")
ax.grid(True, which="both", alpha=0.18, lw=0.5)
ax.legend(loc="lower left", fontsize=8.5, framealpha=0.9)
fig.tight_layout()

FIG_PATH = RESULTS_DIR / "figures" / "three_country_phase_points.png"
FIG_PATH.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(FIG_PATH, dpi=300, bbox_inches="tight")
print(f"Flagship figure draft saved: {FIG_PATH} ({n_clip} points clamped)")
plt.show()

## Readout: Confusion Matrix and Placement Verification (all numbers transcribed from the DE script's output, nothing hand-typed)

In [ ]:
# Combined three-country confusion matrix and per-country accuracy (transcribed from summary + cross-checked by recomputing from the CSV)
tc = summary["three_country"]
cm = tc["confusion_matrix_total"]
print("Combined three-country confusion matrix (theoretical prediction x actual DeltaRMSE sign):")
print(f"  help/help={cm['pred_help_actual_help']}, "
      f"help/hurt={cm['pred_help_actual_hurt']}, "
      f"hurt/help={cm['pred_hurt_actual_help']}, "
      f"hurt/hurt={cm['pred_hurt_actual_hurt']}")
print(f"  Total {cm['total']} points, overall accuracy {tc['total_accuracy']:.3f}")

acc_recomputed = float((three["predicted_help"].astype(bool)
                        == three["actual_help"].astype(bool)).mean())
assert abs(acc_recomputed - tc["total_accuracy"]) < 1e-12

print("\nPer-country accuracy:")
for country in ("UK", "AU", "DE"):
    cc = tc["confusion_by_country"][country]
    print(f"  {country}: {cc['accuracy']:.3f} (n={cc['total']})")

ts = tc["gnn_np_three_sides"]
print(f"\nThree placements on the same theory line (GNN x NP arm, value = min(predicted fraction, actual fraction)):")
print(f"  UK hurt side {ts['uk_gnn_np_hurt_value']:.3f} AND "
      f"AU help side {ts['au_gnn_np_help_value']:.3f} AND "
      f"DE hurt side {ts['de_gnn_np_hurt_value']:.3f} "
      f"-> value={ts['value']:.3f} [{ts['verdict']}]")

g = summary["de_gnn_mult_in_hurt_region"]
s = summary["de_static_in_help_region"]
print(f"\nDE GNN x multiplicative (9 rows) falls in hurt region: value={g['value']:.3f} [{g['verdict']}], "
      f"median (ρ, σ_ratio) = ({g['median_pearson_log']:.3f}, "
      f"{g['median_sigma_ratio']:.3f})")
print(f"DE static x multiplicative (6 rows) falls in help region: value={s['value']:.3f} [{s['verdict']}]")

ss = summary["small_sample_registration"]
print(f"\nSmall-sample registration: n={ss['n_substations']} stations (single region), "
      f"Fisher z SE={ss['fisher_z_se']:.3f}, "
      f"DE GNN x NP median rho={ss['example_de_gnn_np_median_rho']:.3f} "
      f"(95% CI [{ss['example_de_gnn_np_median_rho_ci95'][0]:.3f}, "
      f"{ss['example_de_gnn_np_median_rho_ci95'][1]:.3f}])")

In [ ]:
# DE's 15-point detail table (4-decimal display; full precision in de_phase_points.csv)
de_view = three[three["country"] == "DE"][
    ["base_type", "seed", "signal", "pearson_log", "sigma_ratio",
     "theory_threshold", "delta_rmse", "predicted_help", "actual_help",
     "prediction_correct"]].reset_index(drop=True)
num_cols = ["pearson_log", "sigma_ratio", "theory_threshold", "delta_rmse"]
de_view[num_cols] = de_view[num_cols].round(4)
de_view

## Conclusion Highlights (qualitative; all numbers generated by rule in the DE script, see the output above and `de_phase_summary.json`)

1. **The three countries are three placements on the same theory line**: the UK's GNN x
   multiplicative falls on the hurt side, AU's falls on the help side, and DE's falls on
   the hurt side -- the "replication/non-replication" across the three countries is
   uniformly predicted by the conditional proposition rho > sigma_ratio/2, rather than
   indicating the theory fails in any one country.
2. **DE GNN x multiplicative**: sigma_ratio is high (the correction perturbation is too
   large relative to the base residual) while rho is insufficient to compensate -> falls
   in the hurt region; the theoretical prediction is directionally consistent with the
   observed worsening (GNNpostN +64% / GNNpostNP +193%) (8 of 9 rows predicted correctly;
   1 row, seed 456 x N, is a borderline misclassification -- rho is just above the
   threshold but the outcome still worsened).
3. **DE static x multiplicative**: sigma_ratio is low and rho is high -> falls in the help
   region and all outcomes actually improved (structurally the same as the UK's P1
   proposition).
4. **Small samples are registered as-is**: each DE placement's rho is estimated from only
   13 stations (single region), Fisher z SE ~= 0.316 -- exactly the small-sample regime
   that a robustness discriminator would flag; DE placements are used in the paper only
   for their qualitative direction, not for statistical inference.